<a href="https://colab.research.google.com/github/JuliPerezE/SRA-TP-Filtros-Grupo5/blob/main/notebooks/03_Sensibilidad_y_Validacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sympy as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def sensibilidad_normalizada(y, x):
    return sp.simplify(x * sp.diff(y, x) / y)

### Ejemplo guiado: frecuencia natural de segundo orden
# Para \omega_0=(R_1R_2C_1C_2)^{-1/2} calcular S_x^{\omega_0}=(x/\omega_0)\partial\omega_0/\partial x antes de imponer componentes iguales.
R1, R2, C1, C2 = sp.symbols('R1 R2 C1 C2', positive=True)
w0 = 1 / sp.sqrt(R1 * R2 * C1 * C2)
print("--- Ejemplo guiado: Sensibilidades de w0 ---")
for x in [R1, R2, C1, C2]:
    print(f"S_{x}^w0 =", sensibilidad_normalizada(w0, x))

# ==============================================================================
# ACTIVIDAD 1 (obligatoria) — Sensibilidades reales de la topología Sallen-Key
# ==============================================================================
# Definición simbólica incluyendo la ganancia K del operacional
R1, R2, C1, C2, K = sp.symbols('R1 R2 C1 C2 K', positive=True)

# Expresiones algebraicas reales obtenidas de la celda Sallen-Key pasabajos
w0_real = 1 / sp.sqrt(R1 * R2 * C1 * C2)
Q_real  = sp.sqrt(R1 * R2 * C1 * C2) / (C1 * R1 * (1 - K) + C2 * (R1 + R2))

# Evaluación para la configuración buffer empleada (K = 1)
Q_real_k1 = Q_real.subs(K, 1)

# Componentes independientes de la celda de 2.º orden
componentes = [R1, R2, C1, C2]

# Valores de componentes comerciales seleccionados en la Actividad 2
valores_comerciales = {
    R1: 6800.0,   # 6.8 kΩ (Serie E24)
    R2: 16000.0,  # 16 kΩ (Serie E24)
    C1: 22e-9,    # 22 nF (Serie E24)
    C2: 10e-9     # 10 nF (Serie E24)
}

# Construcción de la tabla de sensibilidades simbólicas y numéricas
filas_sens = []
for x in componentes:
    # Sensibilidad de w0
    s_w0_sym = sensibilidad_normalizada(w0_real, x)
    s_w0_val = float(s_w0_sym.subs(valores_comerciales))

    # Sensibilidad de Q (evaluada en K = 1)
    s_q_sym = sensibilidad_normalizada(Q_real_k1, x)
    s_q_val = float(s_q_sym.subs(valores_comerciales))

    filas_sens.append({
        'Componente': str(x),
        'Valor Comercial': f"{valores_comerciales[x]/1e3:.1f} kΩ" if 'R' in str(x) else f"{valores_comerciales[x]*1e9:.0f} nF",
        'S_x(w0) Simbólica': str(s_w0_sym),
        'S_x(w0) Eval.': f"{s_w0_val:+.4f}",
        'S_x(Q) Simbólica': str(s_q_sym),
        'S_x(Q) Eval.': f"{s_q_val:+.4f}"
    })

sensabilidades = pd.DataFrame(filas_sens)

print("\n" + "=" * 80)
print("📋 ACTIVIDAD 1: SENSIBILIDADES REALES EVALUADAS EN COMPONENTES COMERCIALES")
print("=" * 80)
display(sensabilidades)